# Multi-Domain Retrieval Pipeline

迭代式多域蛋白质生成：
1. DOMIN 检索与 query_domain 匹配的下一个 domain
2. DOMO 基于原始 domain 列表合成序列

In [18]:
%load_ext autoreload
%autoreload 2

ROOT_DIR = "/sujin/PycharmProjects/DOMINO"

import sys
sys.path.append(ROOT_DIR)
sys.path.append(f"{ROOT_DIR}/src/DOMO")
sys.path.append(f"{ROOT_DIR}/src/DOMO/models")

import logging
import torch
import faiss
import numpy as np
from tqdm import tqdm
import yaml

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 1. 模型初始化

In [19]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [20]:
# 初始化 DOMIN 模型
from src.DOMIN.models.ted.ted_domain_model import TedDomainModel

config_path = f"{ROOT_DIR}/weights/DOMIN"
domin_checkpoint = f"{ROOT_DIR}/weights/DOMIN/DOMIN.pt"
model_config = {
    "config_path": config_path,
    "from_checkpoint": domin_checkpoint,
}

domin_model = TedDomainModel(**model_config)
domin_model.to(device).eval()
print(f"DOMIN model loaded. Temperature: {domin_model.model.temperature.item():.4f}")

No lr_scheduler_kwargs provided. The default learning rate is 0.
No optimizer_kwargs provided. The default optimizer is AdamW.
Some weights of the model checkpoint were not used: ['esm.embeddings.position_ids']
DOMIN model loaded. Temperature: 0.0051


In [21]:
# 初始化 DOMO 模型
from src.DOMO.utils.init_utils import construct_class_by_name

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

config = {
    "class_name": "models.Qwen3CAwDomainConditioning.Qwen3CAwDomainConditioning",
    "qwen3_type": "/sujin/Models/Qwen/Qwen3-0.6B",
    "esm_type": "/sujin/Models/esm/esm2_t33_650M_UR50D"
}

domo_config_path = f"{ROOT_DIR}/src/DOMO/configs/DOM"

domo_checkpoint = f"{ROOT_DIR}/weights/DOMO/pytorch_model.bin"

domo_model = construct_class_by_name(**config, logger=logger)
domo_model.load_state_dict(torch.load(domo_checkpoint, map_location=device))
domo_model.eval()
domo_model = domo_model.to(device)

tokenizer = domo_model.tokenizer
print("DOMO model loaded successfully")

INFO:__main__:BaseModel initialized with kwargs: {}
INFO:__main__:Qwen3CAwDomainConditioning initialized with gpt_type: /sujin/Models/Qwen/Qwen3-0.6B, esm_type: /sujin/Models/esm/esm2_t33_650M_UR50D
Some weights of EsmModel were not initialized from the model checkpoint at /sujin/Models/esm/esm2_t33_650M_UR50D and are newly initialized: ['contact_head.regression.bias', 'contact_head.regression.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


DOMO model loaded successfully


In [22]:
# 加载 FAISS 索引和序列映射
INDEX_DIR = "/sujin/Datasets/TED/embedding/afdb_cluster_power0.75/"
INDEX_PATH = f"{INDEX_DIR}/key_embeddings.index"
TSV_PATH = f"{INDEX_DIR}/sampled_proteins.tsv"

faiss_index = faiss.read_index(str(INDEX_PATH), faiss.IO_FLAG_MMAP)
print(f"FAISS index loaded: {faiss_index.ntotal:,} vectors, dim={faiss_index.d}")

print(f"Loading sequence mapping from: {TSV_PATH}")

idx_to_segment = []
with open(TSV_PATH, 'r') as f:
    for line in tqdm(f, desc="Loading segments"):
        parts = line.strip().split('\t')
        idx_to_segment.append(parts[1])

print(f"Loaded {len(idx_to_segment):,} segments")
print(f"Example segment (idx=0): {idx_to_segment[0][:80]}...")

FAISS index loaded: 40,782,345 vectors, dim=1280
Loading sequence mapping from: /sujin/Datasets/TED/embedding/afdb_cluster_power0.75//sampled_proteins.tsv


Loading segments: 40782345it [00:42, 966243.98it/s] 

Loaded 40,782,345 segments
Example segment (idx=0): SvTvLlEvEvQvIlHvLvAlEvEvIvIlKvKvDlSvEvTvLdDdKpNvDpRpFvKsAvAvSvFsNvSvFsSvEvHvLsAl...


## 2. 定义 Pipeline 函数

In [35]:
def retrieve_next_domain(query_domain: str, top_k: int = 1, nprobe: int = 1) -> list:
    """
    使用 DOMIN 检索与 query_domain 匹配的下一个 domain
    
    Args:
        query_domain: 查询域的序列
        top_k: 返回前 k 个候选
        nprobe: FAISS 搜索时探查的聚类中心数
    
    Returns:
        List of (index, distance, segment) tuples
    """
    faiss_index.nprobe = nprobe
    
    with torch.no_grad():
        query_repr = domin_model.get_query_repr(query_domain).cpu().numpy()
    
    distances, indices = faiss_index.search(query_repr.reshape(1, -1).astype('float32'), top_k)
    distances /= domin_model.model.temperature.item()
    
    results = []
    for idx, dist in zip(indices[0].tolist(), distances[0].tolist()):
        if idx != -1 and dist != -1:
            segment = idx_to_segment[idx] if idx < len(idx_to_segment) else None
            results.append((idx, dist, segment))
    
    return results

print("retrieve_next_domain() function defined")

retrieve_next_domain() function defined


In [39]:
def generate_combined_sequence(domain_list: list) -> str:
    """
    使用 DOMO 将多个 domain 合成一个序列
    
    Args:
        domain_list: domain 序列列表
    
    Returns:
        合成的蛋白质序列
    """
    print(domain_list)
    tokenized_domain = tokenizer(domain_list, return_tensors="pt", padding=True, truncation=True, max_length=512)
    domain_ids = tokenized_domain.input_ids.to(device)
    domain_masks = tokenized_domain.attention_mask.to(device)
    num_domains_per_protein = torch.tensor([len(domain_list)]).to(device)
    
    with torch.no_grad():
        result = domo_model.generate(
            domain_ids=domain_ids,
            domain_masks=domain_masks,
            num_domains_per_protein=num_domains_per_protein
        )
    return result["output_seqs"][0]

print("generate_combined_sequence() function defined")
example_domains = [
    "MTAFQKLDFSVNDVIESVKDGNVIGRGGAGVVYHGKTPNGVEIAVKKLMGFNGINGHDHGFKAEIRTLGNIRHRNIVRLLAFCSNKDTNLLVYDYMRNGSLGEALHGKKGGILGWNLRYKIAVDAAKGLCYLHHDCEPLIVHRDVKSNNILLDSSFEARVADFGLAKFL",
    "SGNNFSGPIPPSIGQLRQVVKIDLSGNSLEARIPLEIGNCLHLNYLDLSKNELSGSIPQEISDIKILNYLNLSRNHLNDTIPQSIASMKSLTTVDFSYNDLAGKLPETGQFSVFNATSFIGNPRLCGPLLN"
]
combined_seq = generate_combined_sequence(example_domains)
print(f"Combined sequence: {combined_seq}")

generate_combined_sequence() function defined
['MTAFQKLDFSVNDVIESVKDGNVIGRGGAGVVYHGKTPNGVEIAVKKLMGFNGINGHDHGFKAEIRTLGNIRHRNIVRLLAFCSNKDTNLLVYDYMRNGSLGEALHGKKGGILGWNLRYKIAVDAAKGLCYLHHDCEPLIVHRDVKSNNILLDSSFEARVADFGLAKFL', 'SGNNFSGPIPPSIGQLRQVVKIDLSGNSLEARIPLEIGNCLHLNYLDLSKNELSGSIPQEISDIKILNYLNLSRNHLNDTIPQSIASMKSLTTVDFSYNDLAGKLPETGQFSVFNATSFIGNPRLCGPLLN']
{'output_seqs': ['MISGNNFSGPIPPSIGQLRQVVKIDLSGNSLEARIPLEIGNCLHLNYLDLSKNELSGSIPQEISDIKILNYLNLSRNHLNDTIPQSIASMKSLTTVDFSYNDLAGKLPETGQFSVFNATSFIGNPRLCGPLLNSCSSDGVTHAPSPAVSPGEFPSTLTDGGGRLGLKWKMTAFQKLDFSVNDVIESVKDGNVIGRGGAGVVYHGKTPNGVEIAVKKLMGFNGINGHDHGFKAEIRTLGNIRHRNIVRLLAFCSNKDTNLLVYDYMRNGSLGEALHGKKGGILGWNLRYKIAVDAAKGLCYLHHDCEPLIVHRDVKSNNILLDSSFEARVADFGLAKFLGEDVTHVSTQVRGTMGYVAPGFT']}
Combined sequence: MISGNNFSGPIPPSIGQLRQVVKIDLSGNSLEARIPLEIGNCLHLNYLDLSKNELSGSIPQEISDIKILNYLNLSRNHLNDTIPQSIASMKSLTTVDFSYNDLAGKLPETGQFSVFNATSFIGNPRLCGPLLNSCSSDGVTHAPSPAVSPGEFPSTLTDGGGRLGLKWKMTAFQKLDFSVNDVIESVKDGNVIGRGGAGVVYHGKTPNGVEIAVKKLMGFNGINGHDHGFKAEIRTLGNIRHR

## 3. 示例：迭代生成多域蛋白质

In [28]:
# 示例初始 query domain
example_query_domain = "SvTvLlEvEvQvIlHvLvAlEvEvIvIlKvKvDlSvEvTvLdDdKpNvDpRpFvKsAvAvSvFsNvSvFsSvEvHvLsAlNvLsRlKvQvLlYqDvLvQlLvRvRv"

domain_pool = [example_query_domain.replace()]

print(f"Initial query domain: {example_query_domain}")
print(f"Domain length (AA): {len(example_query_domain.replace('<unk>', '')) // 2}")

Initial query domain: SvTvLlEvEvQvIlHvLvAlEvEvIvIlKvKvDlSvEvTvLdDdKpNvDpRpFvKsAvAvSvFsNvSvFsSvEvHvLsAlNvLsRlKvQvLlYqDvLvQlLvRvRv
Domain length (AA): 53


In [36]:
# === 迭代 1: 检索第一个 target domain ===
print("\n" + "="*60)
print("迭代 1: DOMIN 检索")
print("="*60)

retrieved_1 = retrieve_next_domain(example_query_domain, top_k=1, nprobe=1000)
print(f"Top 5 retrieved domains:")
for i, (idx, dist, seg) in enumerate(retrieved_1):
    seg_preview = seg
    print(f"  {i+1}. idx={idx}, dist={dist:.4f}, seg={seg_preview}")

target_idx_1, target_dist_1, target_seg_1 = retrieved_1[0]
domain_pool.append(target_seg_1)

print(f"\nSelected: idx={target_idx_1}, distance={target_dist_1:.4f}")


迭代 1: DOMIN 检索
Top 5 retrieved domains:
  1. idx=27193889, dist=20.3604, seg=IkEaFfEqSkIkEkAwIkVfKfTaAlSdRpTfKwFkFkIkEwFkYdSkTpCfLaEpEdFiKdKiSfFaEtNfDpSdQwSdSpDpNdInNrFiLiKiVtQmWdSgSsRvQrLdPdTiLgKtPgIpLdSqEaIlEqYvLqQqDpQmHkLiLkLmTfViKaSgLpDpGpYrErSgYrGfEiCdViIdAgLcKnSvMqIeGdSpTfAwQdQkFdLkTdYfLtAdHdRsGsEhEtTgGtNiImRmGgStMmKhViRyVhPdApEvRrMhGdTyRdEd

Selected: idx=27193889, distance=20.3604


In [ ]:
# === DOMO 合成 2-domain 蛋白质 ===
print("\n" + "="*60)
print("DOMO 合成 2-domain 蛋白质")
print("="*60)

print(f"Domain pool for synthesis: {len(domain_pool)} domains")
for i, d in enumerate(domain_pool):
    print(f"  Domain {i+1}: {d[:40]}...")

sequence_2dom = generate_combined_sequence(domain_pool)
print(f"\n合成结果 (2-domain):")
print(f"Sequence: {sequence_2dom}")
print(f"Length: {len(sequence_2dom)} AA")

In [ ]:
# === 迭代 2: 用合成的 2-domain 序列检索第三个 domain ===
print("\n" + "="*60)
print("迭代 2: DOMIN 检索")
print("="*60)

query_for_iter2 = sequence_2dom
print(f"Query (synthesized 2-domain): {query_for_iter2[:50]}...")

retrieved_2 = retrieve_next_domain(query_for_iter2, top_k=5, nprobe=10)
print(f"\nTop 5 retrieved domains:")
for i, (idx, dist, seg) in enumerate(retrieved_2):
    seg_preview = seg[:40] + "..." if seg and len(seg) > 40 else seg
    print(f"  {i+1}. idx={idx}, dist={dist:.4f}, seg={seg_preview}")

target_idx_2, target_dist_2, target_seg_2 = retrieved_2[0]

if target_seg_2 not in domain_pool:
    domain_pool.append(target_seg_2)
    print(f"\nSelected new domain: idx={target_idx_2}, distance={target_dist_2:.4f}")
else:
    print(f"\nTop 1 is already in pool, checking next...")
    for idx, dist, seg in retrieved_2[1:]:
        if seg not in domain_pool:
            domain_pool.append(seg)
            print(f"Selected: idx={idx}, distance={dist:.4f}")
            break

print(f"\nDomain pool now has {len(domain_pool)} domains")

In [ ]:
# === DOMO 合成 3-domain 蛋白质 ===
print("\n" + "="*60)
print("DOMO 合成 3-domain 蛋白质")
print("="*60)

print(f"Domain pool for synthesis: {len(domain_pool)} domains")
for i, d in enumerate(domain_pool):
    print(f"  Domain {i+1}: {d[:40]}...")

sequence_3dom = generate_combined_sequence(domain_pool)
print(f"\n合成结果 (3-domain):")
print(f"Sequence: {sequence_3dom}")
print(f"Length: {len(sequence_3dom)} AA")

## 4. 完整 Pipeline 函数

In [ ]:
def iterative_domain_synthesis(
    initial_domain: str,
    num_iterations: int = 3,
    nprobe: int = 10,
    verbose: bool = True
) -> dict:
    """
    迭代式多域蛋白质合成
    
    Args:
        initial_domain: 初始 domain 序列
        num_iterations: 迭代次数
        nprobe: FAISS 检索参数
        verbose: 是否打印详细信息
    
    Returns:
        dict: 包含 'domain_pool', 'final_sequence', 'iterations' 等信息
    """
    domain_pool = [initial_domain]
    synthesis_history = []
    
    for i in range(1, num_iterations):
        if verbose:
            print(f"\n{'='*60}")
            print(f"迭代 {i}: DOMIN 检索")
            print(f"{'='*60}")
        
        if i == 1:
            query = initial_domain
        else:
            query = synthesis_history[-1]['sequence']
        
        retrieved = retrieve_next_domain(query, top_k=5, nprobe=nprobe)
        
        selected = None
        for idx, dist, seg in retrieved:
            if seg not in domain_pool:
                selected = {'idx': idx, 'dist': dist, 'seg': seg}
                break
        
        if selected is None:
            if verbose:
                print("Warning: 所有检索到的 domain 已在 pool 中")
            break
        
        domain_pool.append(selected['seg'])
        
        if verbose:
            print(f"Selected domain idx={selected['idx']}, dist={selected['dist']:.4f}")
            print(f"Domain pool size: {len(domain_pool)}")
        
        if verbose:
            print(f"\n{'='*60}")
            print(f"DOMO 合成 {len(domain_pool)}-domain 蛋白质")
            print(f"{'='*60}")
        
        sequence = generate_combined_sequence(domain_pool)
        
        synthesis_history.append({
            'iteration': i,
            'domain_pool': domain_pool.copy(),
            'selected_domain': selected,
            'sequence': sequence
        })
        
        if verbose:
            print(f"Synthesized sequence: {sequence[:50]}...")
            print(f"Length: {len(sequence)} AA")
    
    return {
        'domain_pool': domain_pool,
        'final_sequence': synthesis_history[-1]['sequence'] if synthesis_history else initial_domain,
        'iterations': synthesis_history
    }

print("iterative_domain_synthesis() function defined")